In [29]:
import numpy as np
from PIL import Image
from typing import Tuple

In [30]:
image_name = "cloud"

image_path = f"origins/{image_name}.jpg"
image_path_save_1 = f"results/task_6/{image_name}_det_kanny.png"
image_path_save_2 = f"results/task_6/{image_name}_det_kanny_lib.png"

## PIL

In [31]:
def calculate_sigma(size: int) -> float:
    """Вычисляет sigma на основе размера ядра, как в OpenCV."""
    return 0.3 * ((size - 1) * 0.5 - 1) + 0.8

In [32]:
def auto_canny(image: np.ndarray, sigma: float = 0.33) -> Tuple[float, float]:
    """Вычисляет low_threshold и high_threshold"""
    v = np.median(image)
    lower = int(max(0, (1.0 - sigma) * v))
    upper = int(min(255, (1.0 + sigma) * v))
    return lower, upper

In [33]:
def gaussian_kernel(size: int, sigma: float = 1) -> np.ndarray:
    """Создает гауссово ядро заданного размера."""
    size = int(size) // 2
    x, y = np.mgrid[-size:size+1, -size:size+1]
    normal = 1 / (2.0 * np.pi * sigma**2)
    kernel = np.exp(-((x**2 + y**2) / (2.0 * sigma**2))) * normal
    return kernel / np.sum(kernel) 

In [34]:
def convolve(image: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """Свертка изображения с ядром."""
    kernel_size = kernel.shape[0]
    pad_size = kernel_size // 2
    padded_image = np.pad(image, pad_size, mode='constant')
    output = np.zeros_like(image)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            output[i, j] = np.sum(padded_image[i:i+kernel_size, j:j+kernel_size] * kernel)
            
    return output

In [35]:
def sobel_filters(image: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Применение оператора Собеля для вычисления градиентов."""
    kernel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
    kernel_y = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], dtype=np.float32)

    gradient_x = convolve(image, kernel_x)
    gradient_y = convolve(image, kernel_y)

    gradient_magnitude = np.hypot(gradient_x, gradient_y)
    gradient_direction = np.arctan2(gradient_y, gradient_x)

    return gradient_magnitude, gradient_direction

In [36]:
def non_max_suppression(gradient_magnitude: np.ndarray, gradient_direction: np.ndarray) -> np.ndarray:
    """Подавление немаксимумов."""
    suppressed = np.zeros_like(gradient_magnitude)
    angle = gradient_direction * 180 / np.pi
    angle[angle < 0] += 180

    for i in range(1, gradient_magnitude.shape[0] - 1):
        for j in range(1, gradient_magnitude.shape[1] - 1):
            q = 255
            r = 255

            # Угол 0
            if (0 <= angle[i, j] < 22.5) or (157.5 <= angle[i, j] <= 180):
                q = gradient_magnitude[i, j + 1]
                r = gradient_magnitude[i, j - 1]
            # Угол 45
            elif (22.5 <= angle[i, j] < 67.5):
                q = gradient_magnitude[i + 1, j - 1]
                r = gradient_magnitude[i - 1, j + 1]
            # Угол 90
            elif (67.5 <= angle[i, j] < 112.5):
                q = gradient_magnitude[i + 1, j]
                r = gradient_magnitude[i - 1, j]
            # Угол 135
            elif (112.5 <= angle[i, j] < 157.5):
                q = gradient_magnitude[i - 1, j - 1]
                r = gradient_magnitude[i + 1, j + 1]

            if (gradient_magnitude[i, j] >= q) and (gradient_magnitude[i, j] >= r):
                suppressed[i, j] = gradient_magnitude[i, j]
            else:
                suppressed[i, j] = 0

    return suppressed

In [37]:
def double_threshold(suppressed: np.ndarray, low_threshold: float, high_threshold: float) -> np.ndarray:
    """Двойная пороговая фильтрация."""
    edges = np.zeros_like(suppressed)
    strong = suppressed > high_threshold
    weak = (suppressed >= low_threshold) & (suppressed <= high_threshold)

    edges[strong] = 255
    edges[weak] = 100

    return edges

In [38]:
def hysteresis(edges: np.ndarray) -> np.ndarray:
    """Трассировка контуров с гистерезисом."""
    for i in range(1, edges.shape[0] - 1):
        for j in range(1, edges.shape[1] - 1):
            if edges[i, j] == 100:
                if (edges[i + 1, j - 1] == 255 or edges[i + 1, j] == 255 or edges[i + 1, j + 1] == 255 or
                    edges[i, j - 1] == 255 or edges[i, j + 1] == 255 or
                    edges[i - 1, j - 1] == 255 or edges[i - 1, j] == 255 or edges[i - 1, j + 1] == 255):
                    edges[i, j] = 255
                else:
                    edges[i, j] = 0
    return edges


In [39]:
def canny_edge_detector(image_path: np.ndarray, ksize: int = 5, low_threshold: float = 50, high_threshold: float = 150) -> np.ndarray:
    """Применение детектора Канни для изображения"""
    # Чтение изображения и преобразование в grayscale
    image = Image.open(image_path).convert("L")
    image_array = np.array(image, dtype=np.float32)
    sigma = calculate_sigma(ksize)

    # 1. Сглаживание изображения с помощью гауссова фильтра
    kernel = gaussian_kernel(size=ksize, sigma=sigma)
    smoothed = convolve(image_array, kernel)

    # 2. Вычисление градиентов с помощью оператора Собеля
    gradient_magnitude, gradient_direction = sobel_filters(smoothed)

    # low_threshold, high_threshold = auto_canny(gradient_magnitude)  

    # 3. Подавление не-максимумов
    suppressed = non_max_suppression(gradient_magnitude, gradient_direction)

    # 4. Двойная пороговая фильтрация
    edges = double_threshold(suppressed, low_threshold, high_threshold)

    # 5. Трассировка контуров с гистерезисом
    edges = hysteresis(edges)

    return edges

In [40]:
kanny_image = canny_edge_detector(image_path, 5, 50, 100)

Image.fromarray(kanny_image.astype(np.uint8)).save(image_path_save_1)

## Библиотека

In [41]:
import cv2
import numpy as np

image = cv2.imread(image_path)

gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

blurred_image = cv2.GaussianBlur(gray_image, (5, 5), 0)

edges = cv2.Canny(blurred_image, threshold1=50, threshold2=150)

cv2.imwrite(image_path_save_2, edges)

True